In [1]:
from tensorflow.keras.layers import Flatten, Dense, Conv2D, MaxPooling2D, Dropout
from tensorflow.keras.applications.resnet_v2 import ResNet152V2
from tensorflow.keras import applications
from tensorflow.keras.models import Model, Sequential
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow import device
from matplotlib import pyplot
from PIL import ImageFile
import sys
from keras_preprocessing.image import ImageDataGenerator
import os
import tensorflow as tf

In [2]:


# def min_image_count():
#     root = "./Train/"
#     datasets = ["train/", "test/"]
#     features = ["heel_strike","other","toe_off"]

#     train_count = None
#     test_count = None

#     for i, dataset in enumerate(datasets):

#         val = 1000
#         for feature in features:
#             dir_path = root + dataset + feature
#             files = os.listdir(dir_path)
#             image_count = len(files)
#             print(dataset, feature, image_count)

#             if image_count < val:
#                 val = image_count - 1

#         if i == 0:
#             train_count = val
#         elif i == 1:
#             test_count = val

#     return [train_count, test_count]




# min_image_count()




def get_generators(shuffle):
    train_datagen = ImageDataGenerator(rescale=1.0 / 255.0, validation_split=0.2)
    test_datagen = ImageDataGenerator(rescale=1.0 / 255.0, validation_split=0.2)

    if shuffle:
    
        train_datagen = ImageDataGenerator(rescale=1.0/255.0, width_shift_range=0.1,
                                           height_shift_range=0.1, horizontal_flip=True, validation_split=0.2)

    train_it = train_datagen.flow_from_directory("C:/Gait/Train/", class_mode="categorical",
                                                 batch_size=32, target_size=(200, 200), shuffle=shuffle, subset='training')

    test_it = test_datagen.flow_from_directory("C:/Gait/Train/", class_mode="categorical", batch_size=32,
                                               target_size=(200, 200), shuffle=shuffle, subset='validation')

    return train_it, test_it



# ResNet152

In [3]:


optimizer = tf.keras.optimizers.Nadam(learning_rate=0.001, beta_1=0.9,beta_2=0.999)

model = ResNet152V2(include_top=False, input_shape=(200, 200, 3))
for layer in model.layers:
    layer.trainable = False

dropout = Dropout(0.1)(model.layers[-1].output)
flat = Flatten()(dropout)
dense = Dense(128, activation="relu", kernel_initializer="he_uniform")(flat)
dropout = Dropout(0.05)(dense)
dense = Dense(64)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(48)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(36)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(24)(dropout)
dense = Dense(12)(dense)
output = Dense(3, activation="softmax")(dense)

model = Model(inputs=model.inputs, outputs=output)
model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics="accuracy")





ImageFile.LOAD_TRUNCATED_IMAGES = True



es = EarlyStopping(monitor="val_accuracy", min_delta=0.005, verbose=1, patience=2, restore_best_weights=True)

train_it, test_it = get_generators(True)

history = model.fit(train_it, steps_per_epoch=len(train_it), callbacks=[es],
                    validation_data=test_it, validation_steps=len(test_it), epochs=10, verbose=1)

_, acc = model.evaluate(test_it, steps=len(test_it), verbose=0)

model.save("C:/Gait/models/ResNet152V2.h5")
print("Finished training")
print('> %.3f' % (acc * 100.0))


Found 2246 images belonging to 3 classes.
Found 560 images belonging to 3 classes.
Epoch 1/10
71/71 [==============================] - 276s 4s/step - loss: 2.1350 - accuracy: 0.7186 - val_loss: 0.5273 - val_accuracy: 0.7821
Epoch 2/10
71/71 [==============================] - 239s 3s/step - loss: 0.5987 - accuracy: 0.7694 - val_loss: 0.5488 - val_accuracy: 0.7750
Epoch 3/10
71/71 [==============================] - 227s 3s/step - loss: 0.5671 - accuracy: 0.7707 - val_loss: 0.5432 - val_accuracy: 0.7804
Epoch 3: early stopping
Finished training
> 78.214


# InceptionV3

In [4]:

optimizer = tf.keras.optimizers.Nadam(learning_rate=0.001, beta_1=0.9,beta_2=0.999)
model = applications.inception_v3.InceptionV3(include_top=False, input_shape=(200, 200, 3))
for layer in model.layers:
    layer.trainable = False

dropout = Dropout(0.1)(model.layers[-1].output)
flat = Flatten()(dropout)
dense = Dense(128, activation="relu", kernel_initializer="he_uniform")(flat)
dropout = Dropout(0.05)(dense)
dense = Dense(64)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(48)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(36)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(24)(dropout)
dense = Dense(12)(dense)
output = Dense(3, activation="softmax")(dense)

model = Model(inputs=model.inputs, outputs=output)
model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics="accuracy")




ImageFile.LOAD_TRUNCATED_IMAGES = True


es = EarlyStopping(monitor="val_accuracy", min_delta=0.005, verbose=1, patience=2, restore_best_weights=True)

train_it, test_it = get_generators(True)

history = model.fit(train_it, steps_per_epoch=len(train_it), callbacks=[es],
                    validation_data=test_it, validation_steps=len(test_it), epochs=10, verbose=1)

_, acc = model.evaluate(test_it, steps=len(test_it), verbose=0)

model.save("C:/Gait/models/inception_v3.h5")
print("Finished training")
print('> %.3f' % (acc * 100.0))



Found 2246 images belonging to 3 classes.
Found 560 images belonging to 3 classes.
Epoch 1/10
71/71 [==============================] - 58s 754ms/step - loss: 1.5531 - accuracy: 0.7324 - val_loss: 0.6375 - val_accuracy: 0.7750
Epoch 2/10
71/71 [==============================] - 53s 742ms/step - loss: 0.5898 - accuracy: 0.7707 - val_loss: 0.5743 - val_accuracy: 0.7750
Epoch 3/10
71/71 [==============================] - 53s 743ms/step - loss: 0.5563 - accuracy: 0.7725 - val_loss: 0.5458 - val_accuracy: 0.7750
Epoch 3: early stopping
Finished training
> 77.500


# InceptionResNetV2

In [5]:


optimizer = tf.keras.optimizers.Nadam(learning_rate=0.001, beta_1=0.9,beta_2=0.999)
model = applications.InceptionResNetV2(include_top=False, input_shape=(200, 200, 3))
for layer in model.layers:
    layer.trainable = False

dropout = Dropout(0.1)(model.layers[-1].output)
flat = Flatten()(dropout)
dense = Dense(128, activation="relu", kernel_initializer="he_uniform")(flat)
dropout = Dropout(0.05)(dense)
dense = Dense(64)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(48)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(36)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(24)(dropout)
dense = Dense(12)(dense)
output = Dense(3, activation="softmax")(dense)

model = Model(inputs=model.inputs, outputs=output)
model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics="accuracy")




ImageFile.LOAD_TRUNCATED_IMAGES = True


es = EarlyStopping(monitor="val_accuracy", min_delta=0.005, verbose=1, patience=2, restore_best_weights=True)

train_it, test_it = get_generators(True)

history = model.fit(train_it, steps_per_epoch=len(train_it), callbacks=[es],
                    validation_data=test_it, validation_steps=len(test_it), epochs=10, verbose=1)

_, acc = model.evaluate(test_it, steps=len(test_it), verbose=0)

model.save("C:/Gait/models/InceptionResNetV2.h5")
print("Finished training")
print('> %.3f' % (acc * 100.0))


Found 2246 images belonging to 3 classes.
Found 560 images belonging to 3 classes.
Epoch 1/10
71/71 [==============================] - 127s 2s/step - loss: 1.0581 - accuracy: 0.7493 - val_loss: 0.5504 - val_accuracy: 0.7750
Epoch 2/10
71/71 [==============================] - 115s 2s/step - loss: 0.6134 - accuracy: 0.7711 - val_loss: 0.5706 - val_accuracy: 0.7750
Epoch 3/10
71/71 [==============================] - 115s 2s/step - loss: 0.5893 - accuracy: 0.7729 - val_loss: 0.5638 - val_accuracy: 0.7750
Epoch 3: early stopping
Finished training
> 77.500


# Xception

In [6]:

optimizer = tf.keras.optimizers.Nadam(learning_rate=0.001, beta_1=0.9,beta_2=0.999)
model = applications.Xception(include_top=False, input_shape=(200, 200, 3))
for layer in model.layers:
    layer.trainable = False

dropout = Dropout(0.1)(model.layers[-1].output)
flat = Flatten()(dropout)
dense = Dense(128, activation="relu", kernel_initializer="he_uniform")(flat)
dropout = Dropout(0.05)(dense)
dense = Dense(64)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(48)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(36)(dropout)
dropout = Dropout(0.05)(dense)
dense = Dense(24)(dropout)
dense = Dense(12)(dense)
output = Dense(3, activation="softmax")(dense)

model = Model(inputs=model.inputs, outputs=output)
model.compile(optimizer=optimizer, loss="categorical_crossentropy", metrics="accuracy")




ImageFile.LOAD_TRUNCATED_IMAGES = True


es = EarlyStopping(monitor="val_accuracy", min_delta=0.005, verbose=1, patience=2, restore_best_weights=True)

train_it, test_it = get_generators(True)

history = model.fit(train_it, steps_per_epoch=len(train_it), callbacks=[es],
                    validation_data=test_it, validation_steps=len(test_it), epochs=10, verbose=1)

_, acc = model.evaluate(test_it, steps=len(test_it), verbose=0)

model.save("C:/Gait/models/Xception.h5")
print("Finished training")
print('> %.3f' % (acc * 100.0))



Found 2246 images belonging to 3 classes.
Found 560 images belonging to 3 classes.
Epoch 1/10
71/71 [==============================] - 108s 1s/step - loss: 2.0818 - accuracy: 0.7355 - val_loss: 0.5841 - val_accuracy: 0.7750
Epoch 2/10
71/71 [==============================] - 105s 1s/step - loss: 0.5867 - accuracy: 0.7698 - val_loss: 0.5383 - val_accuracy: 0.7750
Epoch 3/10
71/71 [==============================] - 104s 1s/step - loss: 0.5725 - accuracy: 0.7720 - val_loss: 0.5800 - val_accuracy: 0.7750
Epoch 3: early stopping
Finished training
> 77.500
